In [9]:
def get_current_weather(location):
    weather_info = {
        "New York": "Sunny, 25°C",
        "San Francisco": "Foggy, 15°C",
        "Los Angeles": "Cloudy, 20°C",
    }
    return weather_info.get(location, {"Weather": "unknown", "Temperature": 0})




In [10]:
def stock_price(ticker):
    stock_info = {
        "AAPL": "$150",
        "GOOGL": "$2800",
        "AMZN": "$3400",
    }
    return stock_info.get(ticker, {"Price": "unknown"})

In [11]:
functions = {
    "get_current_weather": get_current_weather,
    "stock_price": stock_price,
}

In [12]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather in a given location",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "The city and state, e.g. San Francisco, CA",
                    }
                },
                "required": ["location"],
            },
        }
    },
    {
        "type": "function",
        "function": {
            "name": "stock_price",
            "description": "Get the current stock price for a given ticker symbol",
            "parameters": {
                "type": "object",
                "properties": {
                    "ticker": {
                        "type": "string",
                        "description": "The stock ticker symbol, e.g. AAPL",
                    }
                },
                "required": ["ticker"],
            },
        }
    },
]   

In [13]:
from groq import Groq
import os
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

messages = [
    {"role": "user", "content": "What is the current weather in New York?"}
]

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages,
    tools=tools,
    tool_choice="auto"
)

In [14]:
msg = response.choices[0].message

if msg.tool_calls:
    tool_call = msg.tool_calls[0]
    print(tool_call.function.name)
    print(tool_call.function.arguments)

get_current_weather
{"location":"New York, NY"}


In [15]:
import json

args = json.loads(tool_call.function.arguments)

def get_weather(location):
    return f"Weather in {location}: 28°C, sunny"

result = get_weather(args["location"])

In [16]:
messages.append(msg)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": result
})

final = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=messages
)

print(final.choices[0].message.content)

The current weather in New York is 28°C (82°F) with sunny conditions.
